# Identifying Work Tasks in OSHA Accident Narratives with JAAT TaskMatch

**Purpose:** This notebook demonstrates how [JAAT](https://github.com/Job-Ad-Research-at-QSB-LUC/JAAT)'s `TaskMatch` module can extract standardized O\*NET work tasks from unstructured text — not just job advertisements, but any text that describes work activities.

**Key insight:** `TaskMatch` uses semantic similarity via sentence embeddings to match text against ~20,000 O\*NET task descriptions. Because it relies on meaning rather than keywords, it generalizes to any text where work tasks are described — including workplace accident narratives.

**Data:** We use a sample of 1,000 OSHA workplace accident investigation abstracts. Each abstract describes an incident: what the worker was doing, what went wrong, and the outcome. The full dataset contains 111,000+ investigations and is available from [OSHA's public data](https://www.osha.gov/data).

**Pipeline:** Raw text → sentence splitting → TaskMatch → aggregation → visualization

## 1. Setup

In [ ]:
!pip install -q JAAT

import os
if not os.path.exists('data/accident_abstracts_sample.csv'):
    !git clone --depth 1 https://github.com/pnorlander/JAAT_Demos.git
    os.chdir('JAAT_Demos/osha_accident_taskmatch')

import pandas as pd
import matplotlib.pyplot as plt
import time
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from JAAT import JAAT

## 2. Load Data

Each row is one OSHA accident investigation, identified by `summary_nr`. The `abstract_text` field contains the full narrative describing what happened.

In [ ]:
df = pd.read_csv('data/accident_abstracts_sample.csv')
print(f'Loaded {len(df)} accident investigations')
print(f'Average narrative length: {df["abstract_text"].str.len().mean():.0f} characters')
df.head(3)

Let's look at a few examples to understand the data:

In [ ]:
for i in [0, 5, 10]:
    print(f'--- Investigation {df.iloc[i]["summary_nr"]} ---')
    print(df.iloc[i]['abstract_text'])
    print()

## 3. Text Preprocessing: Sentence Splitting

`TaskMatch` works best on individual sentences. We split each accident narrative into sentences using NLTK's sentence tokenizer, creating one row per sentence.

In [ ]:
# Split each narrative into sentences
assert df['summary_nr'].is_unique, 'Duplicate summary_nr values found — check data'

df['sentences'] = df['abstract_text'].apply(sent_tokenize)
sentences_df = df.explode('sentences').reset_index(drop=True)
sentences_df['sentence_nr'] = sentences_df.groupby('summary_nr').cumcount() + 1
sentences_df = sentences_df[['summary_nr', 'sentence_nr', 'sentences']].copy()

print(f'{len(df)} narratives → {len(sentences_df)} sentences')
print(f'Average sentences per narrative: {len(sentences_df) / len(df):.1f}')
sentences_df.head()

**Before and after** — here's one narrative split into its component sentences:

In [ ]:
example_id = df.iloc[0]['summary_nr']
print(f'Original narrative (investigation {example_id}):')
print(df.iloc[0]['abstract_text'])
print(f'\nSplit into {len(sentences_df[sentences_df["summary_nr"] == example_id])} sentences:')
for _, row in sentences_df[sentences_df['summary_nr'] == example_id].iterrows():
    print(f'  {row["sentence_nr"]}: {row["sentences"]}')

## 4. Run TaskMatch

We initialize `TaskMatch` with a threshold of 0.80 (exploratory, lower than the default 0.90) and run it on all sentences. TaskMatch compares each sentence against ~20,000 O\*NET task descriptions using sentence embeddings and returns matches above the similarity threshold.

In [ ]:
# Initialize TaskMatch
task_matcher = JAAT.TaskMatch(threshold=0.80)

# Process all sentences in batches
start = time.time()
texts = sentences_df['sentences'].fillna('').tolist()
matches = task_matcher.get_tasks_batch(texts)
elapsed = time.time() - start

sentences_df['matched_tasks'] = matches
sentences_df['n_matches'] = sentences_df['matched_tasks'].apply(len)

n_with_match = (sentences_df['n_matches'] > 0).sum()
print(f'Processed {len(sentences_df)} sentences in {elapsed:.1f}s')
print(f'Sentences with at least one task match: {n_with_match} ({n_with_match/len(sentences_df)*100:.1f}%)')
print(f'Unique investigations with matches: {sentences_df[sentences_df["n_matches"] > 0]["summary_nr"].nunique()} / {len(df)}')

Let's look at some example matches to see what TaskMatch found:

In [ ]:
# Show sentences with their matched O*NET tasks
matched = sentences_df[sentences_df['n_matches'] > 0].head(10)
for _, row in matched.iterrows():
    print(f'Sentence: "{row["sentences"][:120]}..."' if len(row['sentences']) > 120 else f'Sentence: "{row["sentences"]}"')
    for task_id, task_desc in row['matched_tasks']:
        print(f'  → O*NET Task {task_id}: {task_desc}')
    print()

## 5. Aggregate & Visualize: All Matched Tasks

Let's count the most frequently matched O\*NET tasks across all accident narratives.

In [ ]:
# Flatten all matched tasks into a single list
all_tasks = []
for task_list in sentences_df['matched_tasks']:
    if task_list:
        for task in task_list:
            if isinstance(task, (list, tuple)) and len(task) > 1:
                all_tasks.append((task[0], str(task[1])))

task_counts = pd.Series(all_tasks).value_counts().reset_index()
task_counts.columns = ['task', 'count']
task_counts['task_id'] = task_counts['task'].apply(lambda t: t[0])
task_counts['description'] = task_counts['task'].apply(lambda t: t[1])
task_counts = task_counts[['task_id', 'description', 'count']]

print(f'Total task matches: {len(all_tasks)}')
print(f'Unique O*NET tasks matched: {len(task_counts)}')
task_counts.head(20)

In [ ]:
# Plot top 20 tasks
top20 = task_counts.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(top20)-1, -1, -1), top20['count'], color='#4A90D9')
ax.set_yticks(range(len(top20)-1, -1, -1))
ax.set_yticklabels([d[:80] + '...' if len(d) > 80 else d for d in top20['description']], fontsize=9)
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 20 O*NET Tasks Matched in OSHA Accident Narratives (All Tasks)', fontsize=14)

for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Domain Knowledge Cleanup: Separating Responder Tasks

Notice that several top-ranked tasks describe **emergency response** activities — contacting medical personnel, carrying injured people to safety, performing emergency work. These tasks are matched because accident narratives describe what happened *after* the injury: EMTs arriving, firefighters responding, etc.

These are real matches (the text genuinely describes those tasks), but they reflect the **response to** the accident, not the **work being performed** when the accident occurred. For a workplace safety analysis, we want to separate these.

We identify responder tasks by their O\*NET task IDs and show the before/after comparison.

In [ ]:
# Identify emergency responder tasks by examining the top matches
# These are tasks primarily associated with EMTs, firefighters, police, and hospital workers
responder_keywords = [
    'emergency medical', 'emergency personnel', 'emergency situation',
    'rescue victim', 'carry injured', 'transport patient',
    'administer first aid', 'treat wound', 'resuscit',
    'fire suppression', 'extinguish', 'evacuate',
    'crime scene', 'arrest', 'apprehend',
    'ambulance', 'paramedic',
    'emergency work during off-hours',
]

def is_responder_task(description):
    desc_lower = description.lower()
    return any(kw in desc_lower for kw in responder_keywords)

task_counts['is_responder'] = task_counts['description'].apply(is_responder_task)
n_responder = task_counts[task_counts['is_responder']]['count'].sum()
n_worker = task_counts[~task_counts['is_responder']]['count'].sum()

print(f'Responder task matches: {n_responder} ({n_responder/(n_responder+n_worker)*100:.1f}%)')
print(f'Worker task matches: {n_worker} ({n_worker/(n_responder+n_worker)*100:.1f}%)')
print(f'\nTop responder tasks identified:')
responder_tasks = task_counts[task_counts['is_responder']].head(10)
for _, row in responder_tasks.iterrows():
    print(f'  [{row["count"]:>4}x] {row["description"]}')

In [ ]:
# Side-by-side comparison
worker_tasks = task_counts[~task_counts['is_responder']]
top15_all = task_counts.head(15)
top15_worker = worker_tasks.head(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Left: All tasks
colors_all = ['#D94A4A' if is_responder_task(d) else '#4A90D9' for d in top15_all['description']]
ax1.barh(range(len(top15_all)-1, -1, -1), top15_all['count'], color=colors_all)
ax1.set_yticks(range(len(top15_all)-1, -1, -1))
ax1.set_yticklabels([d[:55] + '...' if len(d) > 55 else d for d in top15_all['description']], fontsize=8)
ax1.set_title('All Matched Tasks', fontsize=13)
ax1.set_xlabel('Frequency')

# Add legend
from matplotlib.patches import Patch
ax1.legend(handles=[
    Patch(facecolor='#4A90D9', label='Worker tasks'),
    Patch(facecolor='#D94A4A', label='Responder tasks'),
], loc='lower right', fontsize=9)

# Right: Worker tasks only
ax2.barh(range(len(top15_worker)-1, -1, -1), top15_worker['count'], color='#4A90D9')
ax2.set_yticks(range(len(top15_worker)-1, -1, -1))
ax2.set_yticklabels([d[:55] + '...' if len(d) > 55 else d for d in top15_worker['description']], fontsize=8)
ax2.set_title('Worker Tasks Only (Responder Tasks Removed)', fontsize=13)
ax2.set_xlabel('Frequency')

plt.suptitle('Effect of Removing Emergency Responder Tasks', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 7. Results: Work Tasks Associated with Workplace Injuries

After removing emergency responder tasks, we see the O\*NET tasks that workers were **performing** when accidents occurred. These represent the work activities most associated with workplace injuries in the OSHA investigation data.

In [ ]:
# Final clean visualization
top20_worker = worker_tasks.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(top20_worker)-1, -1, -1), top20_worker['count'], color='#2E7D32')
ax.set_yticks(range(len(top20_worker)-1, -1, -1))
ax.set_yticklabels([d[:80] + '...' if len(d) > 80 else d for d in top20_worker['description']], fontsize=9)
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 20 Worker Tasks Associated with Workplace Injuries\n(OSHA Accident Narratives, N=1,000 investigations)', fontsize=14)

for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print('=== Summary ===')
print(f'Accident investigations analyzed: {len(df)}')
print(f'Sentences processed: {len(sentences_df)}')
print(f'Sentences with task matches: {n_with_match} ({n_with_match/len(sentences_df)*100:.1f}%)')
print(f'Total task matches: {len(all_tasks)}')
print(f'Unique O*NET tasks identified: {len(task_counts)}')
print(f'Responder tasks filtered: {n_responder} matches across {len(task_counts[task_counts["is_responder"]])} task types')
print(f'Worker tasks retained: {n_worker} matches across {len(worker_tasks)} task types')

## Discussion

This demonstration shows three things:

1. **Semantic similarity generalizes.** JAAT's `TaskMatch` was built to identify O\*NET tasks in job advertisements, but because it uses sentence embeddings rather than keyword matching, it works on any text that describes work activities — including accident narratives written by OSHA investigators.

2. **Domain knowledge matters.** Automated coding tools produce valid matches that still require human interpretation. The emergency responder tasks are genuine semantic matches — the text really does describe those activities — but they describe the *response* to accidents, not the work that caused them. Transparent filtering, as shown above, addresses this.

3. **Text to insight, transparently.** The full pipeline from raw narrative text to structured, visualizable output takes under a minute on this sample. Every step is visible and reproducible.

**Scaling up:** The full OSHA dataset (111,000+ investigations, 538,000+ sentences) processes in approximately 55 minutes on a GPU-equipped machine using `get_tasks_batch()`. Results are available in the accompanying paper.

**Learn more:** [JAAT documentation and source code](https://github.com/Job-Ad-Research-at-QSB-LUC/JAAT)